<a href="https://colab.research.google.com/github/sourish-bhattacharyya/fds-lab/blob/main/Sourish_Bhattacharyya_exp6_FDS_16014225009.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files

uploaded = files.upload()

Saving kolkata_unified_dataset.csv to kolkata_unified_dataset.csv


In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.cluster import KMeans
print(df.head())
print(df.shape)

             Timestamp         Farm_ID  Volumetric_Moisture_Pct  \
0  2025-01-05 00:00:00  Kolkata_Farm_1                    14.15   
1  2025-01-05 00:15:00  Kolkata_Farm_1                    14.14   
2  2025-01-05 00:30:00  Kolkata_Farm_1                    14.14   
3  2025-01-05 00:45:00  Kolkata_Farm_1                    14.14   
4  2025-01-05 01:00:00  Kolkata_Farm_1                    14.14   

   Sensor_Board_Temperature_C  Raw_Capacitive_ADC  Sensor_Voltage_V  \
0                   17.026336                 704             2.271   
1                   17.717099                 692             2.232   
2                   13.012097                 700             2.258   
3                   14.360777                 697             2.248   
4                   15.489267                 675             2.177   

   Atm_Pressure_inHg  Atm_Temperature_C  Soil_Temperature_C  SM_Level_15cm  \
0            30.0206              16.38               19.21           1.80   
1            3

In [8]:
cols = [
    'Volumetric_Moisture_Pct',
    'Soil_Temperature_C',
    'Sensor_Voltage_V'
]

data = df[cols].copy()
print(data.head())

   Volumetric_Moisture_Pct  Soil_Temperature_C  Sensor_Voltage_V
0                    14.15               19.21             2.271
1                    14.14               19.36             2.232
2                    14.14               19.55             2.258
3                    14.14               19.52             2.248
4                    14.14               19.73             2.177


In [9]:
mm = MinMaxScaler()

minmax = pd.DataFrame(
    mm.fit_transform(data),
    columns=[c + '_MinMax' for c in cols]
)

print(minmax.head())

   Volumetric_Moisture_Pct_MinMax  Soil_Temperature_C_MinMax  \
0                        0.488806                   0.226451   
1                        0.487873                   0.233587   
2                        0.487873                   0.242626   
3                        0.487873                   0.241199   
4                        0.487873                   0.251189   

   Sensor_Voltage_V_MinMax  
0                 0.653333  
1                 0.620833  
2                 0.642500  
3                 0.634167  
4                 0.575000  


In [10]:
zs = StandardScaler()

zscore = pd.DataFrame(
    zs.fit_transform(data),
    columns=[c + '_ZScore' for c in cols]
)

print(zscore.head())

   Volumetric_Moisture_Pct_ZScore  Soil_Temperature_C_ZScore  \
0                        1.107528                  -1.387626   
1                        1.102591                  -1.353677   
2                        1.102591                  -1.310675   
3                        1.102591                  -1.317465   
4                        1.102591                  -1.269936   

   Sensor_Voltage_V_ZScore  
0                -0.679994  
1                -0.882813  
2                -0.747601  
3                -0.799606  
4                -1.168840  


In [11]:
decimal = data.copy()

for c in cols:
    maximum = data[c].abs().max()
    j = 0

    while maximum / (10 ** j) >= 1:
        j += 1

    decimal[c] = data[c] / (10 ** j)

decimal.columns = [c + '_Decimal' for c in cols]

print(decimal.head())

   Volumetric_Moisture_Pct_Decimal  Soil_Temperature_C_Decimal  \
0                           0.1415                      0.1921   
1                           0.1414                      0.1936   
2                           0.1414                      0.1955   
3                           0.1414                      0.1952   
4                           0.1414                      0.1973   

   Sensor_Voltage_V_Decimal  
0                    0.2271  
1                    0.2232  
2                    0.2258  
3                    0.2248  
4                    0.2177  


In [12]:
result = pd.concat(
    [data.reset_index(drop=True),
     minmax, zscore,
     decimal.reset_index(drop=True)],
    axis=1
)

print(result.head(10).round(3))

   Volumetric_Moisture_Pct  Soil_Temperature_C  Sensor_Voltage_V  \
0                    14.15               19.21             2.271   
1                    14.14               19.36             2.232   
2                    14.14               19.55             2.258   
3                    14.14               19.52             2.248   
4                    14.14               19.73             2.177   
5                    14.14               19.91             2.203   
6                    14.15               19.59             2.235   
7                    14.12               19.75             2.223   
8                    14.15               19.75             2.229   
9                    14.16               19.72             2.245   

   Volumetric_Moisture_Pct_MinMax  Soil_Temperature_C_MinMax  \
0                           0.489                      0.226   
1                           0.488                      0.234   
2                           0.488                      0.24

In [18]:

kmeans = KMeans(n_clusters=3)

cluster = kmeans.fit_predict(
    data[['Volumetric_Moisture_Pct']]
)

centers = kmeans.cluster_centers_.flatten()
order = np.argsort(centers)

labels = {
    order[0]: 'Low Moisture',
    order[1]: 'Medium Moisture',
    order[2]: 'High Moisture'
}

In [19]:
discretized = pd.DataFrame({
    'Volumetric_Moisture_Pct':
        data['Volumetric_Moisture_Pct'],
    'Cluster': cluster
})

discretized['Moisture_Category'] = (
    discretized['Cluster'].map(labels)
)

print(discretized.head(10))

   Volumetric_Moisture_Pct  Cluster Moisture_Category
0                    14.15        1   Medium Moisture
1                    14.14        1   Medium Moisture
2                    14.14        1   Medium Moisture
3                    14.14        1   Medium Moisture
4                    14.14        1   Medium Moisture
5                    14.14        1   Medium Moisture
6                    14.15        1   Medium Moisture
7                    14.12        1   Medium Moisture
8                    14.15        1   Medium Moisture
9                    14.16        1   Medium Moisture


In [20]:
print(discretized['Moisture_Category'].value_counts())

Moisture_Category
Low Moisture       11820
Medium Moisture     7154
High Moisture       2338
Name: count, dtype: int64
